# 🔍 AI Output Validation Framework

> *A systematic approach to evaluating LLM response quality across 6 critical dimensions.*

**Author:** Albina Urubkina — AI Product Analyst | Prompt Engineering | UX Research  
**Stack:** Python · Anthropic Claude API · LLM-as-Judge pattern

---

## Why this matters

When deploying LLMs in production (customer support, document generation, analytics), you can't manually review every output. This framework uses **Claude as an evaluator (LLM-as-Judge)** to automatically score AI-generated responses across dimensions that matter in real business contexts:

| Dimension | What it checks |
|-----------|----------------|
| **Consistency** | No internal contradictions |
| **Completeness** | All prompt requirements addressed |
| **Tone of Voice** | Matches expected register and style |
| **Hallucination Absence** | No fabricated facts or false claims |
| **Business Logic** | Aligns with domain rules and constraints |
| **Structure & Format** | Correct output format and schema |

Each dimension is scored 0–10. Overall pass threshold: **≥ 7.0**.

## 0. Setup

In [ ]:
# Install dependencies (run once)
# !pip install anthropic python-dotenv

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from validator import validate, batch_validate, generate_report, VALIDATION_DIMENSIONS
import json

# Set your API key
# Option 1: set ANTHROPIC_API_KEY in your .env file
# Option 2: os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

print('✅ Framework loaded successfully')
print(f'Available dimensions: {VALIDATION_DIMENSIONS}')

---
## 1. Single Validation — Basic Usage

Validate one LLM output. You provide the original prompt, the AI response, and optional context.

In [ ]:
prompt = """Write a short product description for a B2B SaaS analytics dashboard.
Target audience: CFOs and finance directors.
Tone: professional, concise, data-driven.
Include key benefits as no more than 3 bullet points."""

ai_output = """Introducing DataLens — your intelligent financial command center.

DataLens empowers finance leaders with:
• Real-time cash flow visibility across all business units
• Automated variance analysis with AI-driven root cause detection  
• One-click board-ready reports in IFRS and GAAP formats

Trusted by 500+ CFOs across Europe. Free trial — no credit card required.
DataLens was founded in 2019 in Berlin and has raised $12M in Series A funding."""

result = validate(
    prompt=prompt,
    ai_output=ai_output,
    context="B2B SaaS product in financial analytics",
    tone_target="formal, professional, data-driven, no fluff",
    business_rules="Max 3 bullet points. No company history. Must be factual."
)

print(result.summary())

---
## 2. Testing the Hallucination Detector

Pass an output that contains fabricated information. Watch the `hallucination_absence` score drop.

In [ ]:
prompt_refund = "Answer: what is the refund policy for our SaaS product?"

# This output invents details that contradict the business rules
ai_output_hallucinated = """Our refund policy allows full refunds within 30 days of purchase,
as stated in our Terms of Service updated in March 2024.
You can also get a partial refund after 60 days if you contact our CEO directly at ceo@company.com."""

result_hallucination = validate(
    prompt=prompt_refund,
    ai_output=ai_output_hallucinated,
    context="Customer support bot for SaaS product",
    business_rules="14-day full refund only. No partial refunds exist. Never share personal emails.",
    dimensions=["hallucination_absence", "business_logic", "completeness"]
)

print(result_hallucination.summary())

---
## 3. Tone of Voice Validation

A common failure mode: prompting for 'professional' and getting 'overly casual'.

In [ ]:
prompt_onboarding = "Write a welcome message for a new user of a project management app."

ai_output_casual = """Hey there! 🎉 So pumped you joined us!
Let's crush those deadlines together — you got this!
The dashboard is your new BFF. Hit us up anytime."""

result_tone = validate(
    prompt=prompt_onboarding,
    ai_output=ai_output_casual,
    tone_target="warm and friendly but professional — not overly casual, no slang",
    dimensions=["tone_of_voice", "structure_format", "completeness"]
)

print(result_tone.summary())

---
## 4. Batch Validation + Summary Report

Validate multiple outputs at once — useful for regression testing after prompt changes.

In [ ]:
# Load sample cases from file
with open('examples/sample_cases.json', encoding='utf-8') as f:
    cases = json.load(f)

print(f'Loaded {len(cases)} test cases:')
for i, c in enumerate(cases):
    print(f'  {i+1}. {c["name"]}')

print('\n🔄 Running batch validation...')
results = batch_validate(cases)

# Print individual summaries
for i, r in enumerate(results):
    print(f'\n--- Case {i+1}: {cases[i]["name"]} ---')
    print(r.summary())

# Print aggregate report
print(generate_report(results))

---
## 5. Export Results to JSON

All results are serializable — save them for logging, dashboards, or audit trails.

In [ ]:
output_path = 'validation_results.json'
export_data = [r.to_dict() for r in results]

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print(f'✅ Saved {len(export_data)} results to {output_path}')

# Preview one result
print('\nSample result structure:')
sample = export_data[0]
print(json.dumps({
    'overall_score': sample['overall_score'],
    'passed': sample['passed'],
    'recommendation': sample['recommendation'],
    'dimensions_evaluated': sample['dimensions_evaluated']
}, indent=2))

---
## 6. Custom Validation — Bring Your Own Output

Replace the variables below with your own prompt and output to validate anything.

In [ ]:
# ✏️ Edit these:
MY_PROMPT = "Your prompt here"
MY_OUTPUT = "The AI response you want to evaluate"
MY_CONTEXT = "Optional: describe the product/domain"
MY_TONE = "Optional: expected tone (e.g. formal, friendly, expert)"
MY_RULES = "Optional: business rules the output must follow"

my_result = validate(
    prompt=MY_PROMPT,
    ai_output=MY_OUTPUT,
    context=MY_CONTEXT,
    tone_target=MY_TONE,
    business_rules=MY_RULES
)

print(my_result.summary())

---

## Architecture Notes

```
User Input (prompt + AI output + context)
        │
        ▼
  validator.py  ──────►  Claude API (Sonnet)
  (judge prompt)              │
        │            Structured JSON response
        ▼
  ValidationResult
  ├── scores (per dimension)
  ├── overall_score
  ├── critical_issues
  ├── recommendation (APPROVE / REVISE / REJECT)
  └── passed (bool, threshold 7.0)
```

**Design decisions:**
- LLM-as-Judge pattern: the same model family that generates can also evaluate (with different system instructions)
- Structured JSON output with strict schema enforcement avoids parsing ambiguity
- Pass/fail threshold at 7.0 allows calibration per use case
- Modular dimensions: pick only the ones relevant to your domain

**Limitations & known issues:**
- Evaluator model can share biases with the evaluated model
- `hallucination_absence` requires accurate `business_rules` / `context` to work well
- API costs scale with batch size — consider caching for large regression suites